In [7]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.datasets import fashion_mnist
from keras.models import Sequential
from keras.layers import Dense, Flatten, Input, Conv2D, MaxPool2D, Conv2DTranspose, UpSampling2D
from keras.losses import MeanSquaredError
from keras.optimizers import AdamW

import matplotlib.pyplot as plt

In [5]:
(x_train, _), (x_test, _) = fashion_mnist.load_data()

print(f"Shape of training set: {x_train.shape}")
print(f"Shape of test set: {x_test.shape}")

Shape of training set: (60000, 28, 28)
Shape of test set: (10000, 28, 28)


In [6]:
x_train = x_train.astype(np.float32) / 255
x_test = x_test.astype(np.float32) / 255

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(f"Shape of training set: {x_train.shape}")
print(f"Shape of test set: {x_test.shape}")

Shape of training set: (60000, 28, 28, 1)
Shape of test set: (10000, 28, 28, 1)


In [12]:
# Add Normal noise
noise_factor = 0.2

x_train_noisy = x_train + noise_factor * tf.random.normal(shape=x_train.shape)    # -0.2 - 1.2
x_test_noisy = x_test + noise_factor * tf.random.normal(shape=x_test.shape)

x_train_noisy = tf.clip_by_value(x_train_noisy, clip_value_min=0., clip_value_max=1.0)
x_test_noisy = tf.clip_by_value(x_test_noisy, clip_value_min=0., clip_value_max=1.0)

In [13]:
class AutoEncoder(keras.Model):
    def __init__(self):
        super().__init__()
        
        self.encoder = Sequential(
            [
                Input(shape=(28, 28, 1)),
                Conv2D(64, (3, 3), activation='relu', strides=2, padding='same'),   # (14, 14, 64)
                Conv2D(16, (3, 3), activation='relu', strides=2, padding='same'),   # (7, 7, 16)
            ]
        )

        self.decoder = Sequential(
            [
                UpSampling2D(size=(2, 2)),  # (14, 14, 16)
                Conv2D(64, (3, 3), activation='relu', padding='same', strides=1),   # (14, 14, 64)
                UpSampling2D(size=(2, 2)),  #(28, 28, 64)
                Conv2D(1, (3, 3), activation='sigmoid', padding='same', strides=1)  # (28, 28, 1)    
            ]
        )

    def call(self, inputs):
        encod = self.encoder(inputs)
        decod = self.decoder(encod)

        return decod
        

In [14]:
model = AutoEncoder()

In [15]:
model(x_train[0].reshape(-1, 28, 28, 1))

<tf.Tensor: shape=(1, 28, 28, 1), dtype=float32, numpy=
array([[[[0.5       ],
         [0.5       ],
         [0.5       ],
         [0.5       ],
         [0.5       ],
         [0.49990445],
         [0.5003228 ],
         [0.50072944],
         [0.50103104],
         [0.50054044],
         [0.5018264 ],
         [0.49993277],
         [0.5000815 ],
         [0.50076556],
         [0.49862278],
         [0.49575192],
         [0.49630237],
         [0.4962115 ],
         [0.496487  ],
         [0.49943906],
         [0.49915585],
         [0.49857658],
         [0.49842414],
         [0.49994367],
         [0.49997056],
         [0.5000985 ],
         [0.4994635 ],
         [0.49965876]],

        [[0.5       ],
         [0.49999914],
         [0.5000009 ],
         [0.5000041 ],
         [0.5000053 ],
         [0.49978817],
         [0.5006364 ],
         [0.50234187],
         [0.50234246],
         [0.5023935 ],
         [0.5070995 ],
         [0.5037771 ],
         [0.5039555 ],

In [17]:
model.encoder.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 7, 7, 16)       │         9,232 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,872 (38.56 KB)

 Trainable params: 9,872 (38.56 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.decoder.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ up_sampling2d_2 (UpSampling2D)  │ (1, 14, 14, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (1, 14, 14, 64)        │         9,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (1, 28, 28, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (1, 28, 28, 1)         │           577 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,857 (38.50 KB)

 Trainable params: 9,857 (38.50 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(optimizer=AdamW(learning_rate=0.001),
              loss=MeanSquaredError)

In [20]:
history = model.fit(x_train_noisy, x_train, validation_split=0.2, epochs=30, batch_size=128)

Epoch 1/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - loss: 0.0226 - val_loss: 0.0124
Epoch 2/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - loss: 0.0112 - val_loss: 0.0101
Epoch 3/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 54ms/step - loss: 0.0096 - val_loss: 0.0090
Epoch 4/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 18s 48ms/step - loss: 0.0088 - val_loss: 0.0085
Epoch 5/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 19s 50ms/step - loss: 0.0083 - val_loss: 0.0081
Epoch 6/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 19s 51ms/step - loss: 0.0080 - val_loss: 0.0078
Epoch 7/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - loss: 0.0078 - val_loss: 0.0076
Epoch 8/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 52ms/step - loss: 0.0076 - val_loss: 0.0075
Epoch 9/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - loss: 0.0074 - val_loss: 0.0073
Epoch 10/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 44ms/step - loss: 0.0073 - val_loss: 0.0072
Epoch 11/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - loss: 0.0071 - val_loss: 0.0072
Epoch 12/30
375/375 ━━━━━━━━━━